# 1. Classification de textes — intentions en français

**Objectif.** Prédire `meteo`, `transport` ou `restaurant` à partir d'une requête. Corpus intégré, hors ligne et volontairement petit. Les scores illustrent le protocole ; ils ne mesurent pas une performance de production.

## 1. Corpus et contrôle des données

Chaque phrase a une seule étiquette. Le découpage stratifié est fait avant l'apprentissage du vocabulaire TF-IDF.

In [1]:
from collections import Counter
from sklearn.model_selection import train_test_split

data = {
    "meteo": [
        "Quel temps fera-t-il demain ?", "Va-t-il pleuvoir ce soir ?", "Quelle est la température à Rabat ?",
        "Le ciel sera-t-il dégagé lundi ?", "Prévois-tu de la neige cette semaine ?",
        "Donne-moi les prévisions météo de Casablanca", "Y aura-t-il du vent sur la côte ?",
        "Fait-il chaud aujourd'hui ?", "Quel est le risque d'orage ?", "La pluie va-t-elle continuer ?",
        "Combien de degrés fait-il dehors ?", "Quel temps est prévu pour le weekend ?",
        "Est-ce qu'il fera soleil à midi ?", "La météo annonce-t-elle du brouillard ?",
        "Quelle humidité est prévue demain ?", "Le temps sera-t-il froid ce matin ?",
        "Est-ce une journée ensoleillée ?", "Quand les averses vont-elles cesser ?",
    ],
    "transport": [
        "À quelle heure part le prochain train ?", "Je cherche un billet de bus pour Fès",
        "Quel est le trajet du tramway ?", "Le métro est-il en retard ?",
        "Combien coûte le ticket de train ?", "Où se trouve la gare routière ?",
        "Réserve une place dans le car de demain", "Quel quai pour le train de Marrakech ?",
        "À quelle heure arrive le bus ?", "Je veux connaître les horaires du tram",
        "Y a-t-il une correspondance à la gare ?", "Le train direct part-il ce soir ?",
        "Comment aller à l'aéroport en bus ?", "Le dernier métro passe à quelle heure ?",
        "Peux-tu acheter un billet de transport ?", "Quel est le numéro de la ligne de bus ?",
        "Le tram circule-t-il le dimanche ?", "Mon train est-il annulé ?",
    ],
    "restaurant": [
        "Je voudrais réserver une table pour deux", "Quel restaurant sert du couscous ?",
        "Peux-tu me montrer le menu ?", "L'addition s'il vous plaît",
        "Y a-t-il une table libre ce soir ?", "Je cherche un endroit pour dîner",
        "Le restaurant propose-t-il des plats végétariens ?", "Combien coûte le plat du jour ?",
        "Je veux commander un dessert", "À quelle heure ouvre la cuisine ?",
        "Ce café accepte-t-il les réservations ?", "Quels sont les meilleurs plats ici ?",
        "Pouvez-vous apporter la carte ?", "Je souhaite déjeuner près de la place",
        "Le service du dîner commence quand ?", "Est-ce que le menu contient du poisson ?",
        "Je voudrais une soupe et une salade", "Où peut-on manger ce midi ?",
    ],
}
texts = [text for label, phrases in data.items() for text in phrases]
labels = [label for label, phrases in data.items() for _ in phrases]
assert len(texts) == len(set(texts))
print("Corpus :", len(texts), "phrases ; répartition :", dict(Counter(labels)))
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, stratify=labels, random_state=42
)
print("Apprentissage :", len(X_train), "; test :", len(X_test))

Corpus : 54 phrases ; répartition : {'meteo': 18, 'transport': 18, 'restaurant': 18}
Apprentissage : 40 ; test : 14


## 2. Référence et modèle

La référence prédit toujours la classe la plus fréquente. Le modèle utilise des n-grammes de caractères, utiles face aux variantes de mots et aux accents.

In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
model = make_pipeline(
    TfidfVectorizer(analyzer="char", ngram_range=(2, 5), lowercase=True),
    LogisticRegression(max_iter=1000, random_state=42),
).fit(X_train, y_train)

def score(y_true, y_pred):
    return {"accuracy": round(accuracy_score(y_true, y_pred), 3),
            "f1_macro": round(f1_score(y_true, y_pred, average="macro"), 3)}

pred_baseline = baseline.predict(X_test)
pred_model = model.predict(X_test)
print("Référence :", score(y_test, pred_baseline))
print("Modèle    :", score(y_test, pred_model))

Référence : {'accuracy': 0.286, 'f1_macro': 0.148}
Modèle    : {'accuracy': 0.714, 'f1_macro': 0.715}


## 3. Évaluation détaillée

Le F1 macro accorde le même poids aux trois classes. La matrice a les vraies classes en lignes et les prédictions en colonnes.

In [3]:
classes = sorted(data)
print(classification_report(y_test, pred_model, labels=classes, zero_division=0))
print("Ordre des classes :", classes)
print("Matrice de confusion :")
print(confusion_matrix(y_test, pred_model, labels=classes))
print("Erreurs :")
for text, truth, prediction in zip(X_test, y_test, pred_model):
    if truth != prediction:
        print(f"- {text!r} | vrai={truth}, prédit={prediction}")

              precision    recall  f1-score   support

       meteo       0.75      0.75      0.75         4
  restaurant       0.67      0.80      0.73         5
   transport       0.75      0.60      0.67         5

    accuracy                           0.71        14
   macro avg       0.72      0.72      0.71        14
weighted avg       0.72      0.71      0.71        14

Ordre des classes : ['meteo', 'restaurant', 'transport']
Matrice de confusion :
[[3 1 0]
 [0 4 1]
 [1 1 3]]
Erreurs :
- 'À quelle heure ouvre la cuisine ?' | vrai=restaurant, prédit=transport
- 'Le métro est-il en retard ?' | vrai=transport, prédit=meteo
- 'Combien de degrés fait-il dehors ?' | vrai=meteo, prédit=restaurant
- 'Réserve une place dans le car de demain' | vrai=transport, prédit=restaurant


## 4. Quelques requêtes nouvelles

Ce contrôle qualitatif ne remplace pas le test tenu à l'écart.

In [4]:
new_queries = ["Va-t-il neiger demain ?", "Le bus pour Tanger part quand ?", "Une table pour quatre ce midi"]
for query, prediction in zip(new_queries, model.predict(new_queries)):
    print(f"{query} -> {prediction}")

Va-t-il neiger demain ? -> meteo
Le bus pour Tanger part quand ? -> restaurant
Une table pour quatre ce midi -> restaurant


## 5. Limites

Corpus de 54 phrases rédigées pour ce cours ; le test aléatoire partage le même style que l'apprentissage. Pour un cas réel, recueillir des requêtes naturelles, séparer par utilisateur ou période et analyser les erreurs par classe.